# 🧠 Exercícios — Paradigmas de IA: Simbólico, Conexionista e Evolucionário

**Disciplina:** Inteligência Artificial | **Nível:** Introdutório

> Compare os três grandes paradigmas da IA com exemplos práticos simples.


## 1. Sistema Simbólico — Base de Regras

In [ ]:
# Mini sistema especialista baseado em regras simbólicas
# Problema: diagnóstico de humor do usuário

regras = [
    # (condições necessárias, diagnóstico)
    ({"cansado", "sem_apetite", "tristeza"}, "Possível depressão leve — consulte um profissional"),
    ({"estressado", "insônia", "irritado"},   "Ansiedade — pratique técnicas de relaxamento"),
    ({"energético", "feliz", "produtivo"},    "Humor ótimo! Continue assim"),
    ({"cansado", "feliz"},                    "Cansaço normal — descanse um pouco"),
    ({"irritado", "fome"},                    "Hangry (fome+raiva) — hora de comer!"),
]

def diagnosticar(sintomas):
    sintomas = set(sintomas)
    for condicoes, diagnostico in regras:
        if condicoes.issubset(sintomas):
            return diagnostico
    return "Não foi possível classificar — adicione mais informações"

# Teste
casos = [
    ["cansado", "feliz", "produtivo"],
    ["irritado", "fome", "insônia"],
    ["tristeza", "cansado", "sem_apetite"],
    ["feliz", "energético"],
]
print("Diagnóstico simbólico:")
for c in casos:
    print(f"  Sintomas: {c}")
    print(f"  → {diagnosticar(c)}\n")


### 📝 Exercício 1

Adicione **3 novas regras** ao sistema e crie 3 casos de teste que as ativem. Pense em outro domínio se quiser (saúde, clima, recomendação de filme, etc.).

In [ ]:
# ✏️ Adicione suas regras e teste:
minhas_regras = regras + [
    # TODO: adicione 3 novas regras
]

def meu_diagnosticar(sintomas):
    sintomas = set(sintomas)
    for condicoes, diagnostico in minhas_regras:
        if condicoes.issubset(sintomas):
            return diagnostico
    return "Sem diagnóstico"

# Casos de teste:
meus_casos = [
    # TODO
]
for c in meus_casos:
    print(f"Sintomas: {c} → {meu_diagnosticar(c)}")


## 2. Sistema Conexionista — Neurônio Simples (Perceptron)

In [ ]:
import numpy as np

class NeuronioSimples:
    """Um único neurônio com aprendizado por gradiente descendente."""
    
    def __init__(self, n_entradas, taxa_aprendizado=0.1):
        np.random.seed(42)
        self.pesos = np.random.randn(n_entradas) * 0.01
        self.vies  = 0.0
        self.lr    = taxa_aprendizado
    
    def sigmoid(self, z): return 1/(1+np.exp(-z))
    
    def prever(self, X): return self.sigmoid(X @ self.pesos + self.vies)
    
    def treinar(self, X, y, epocas=200):
        historico = []
        for _ in range(epocas):
            y_pred = self.prever(X)
            erro = y_pred - y
            self.pesos -= self.lr * X.T @ erro / len(y)
            self.vies  -= self.lr * erro.mean()
            historico.append(((erro**2).mean()))
        return historico

# Problema: AND lógico
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([0,0,0,1], dtype=float)

neuronio = NeuronioSimples(n_entradas=2)
historico = neuronio.treinar(X, y, epocas=500)

import matplotlib.pyplot as plt
plt.figure(figsize=(8,3))
plt.plot(historico)
plt.xlabel('Época'); plt.ylabel('MSE'); plt.title('Curva de Aprendizado — Neurônio AND'); plt.grid(True); plt.show()

print("Previsões após treinamento:")
for xi, yi in zip(X, y):
    pred = neuronio.prever(xi.reshape(1,-1))[0]
    print(f"  {xi} → pred={pred:.3f}  (esperado={yi})  {'✅' if round(pred)==yi else '❌'}")


### 📝 Exercício 2

Tente treinar o neurônio simples para a função **XOR**. Consegue? Por que não? O que seria necessário?

*Dica: XOR não é linearmente separável.*

In [ ]:
# ✏️ Tente XOR com 1 neurônio:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0,1,1,0], dtype=float)  # XOR

neuronio_xor = NeuronioSimples(n_entradas=2, taxa_aprendizado=0.5)
hist_xor = neuronio_xor.treinar(X_xor, y_xor, epocas=1000)

print("XOR com 1 neurônio:")
for xi, yi in zip(X_xor, y_xor):
    pred = neuronio_xor.prever(xi.reshape(1,-1))[0]
    print(f"  {xi} → pred={pred:.3f}  (esperado={yi})  {'✅' if round(pred)==yi else '❌'}")
print(f"\nMSE final: {hist_xor[-1]:.4f}")
print("Conclusão: um único neurônio NÃO consegue resolver XOR.")


## 3. Sistema Evolucionário — Algoritmo Genético Simplificado

In [ ]:
import random

random.seed(42)

def fitness(individuo, alvo):
    """Conta quantos genes diferem do alvo (minimizar)."""
    return sum(a != b for a, b in zip(individuo, alvo))

def criar_populacao(tamanho, comprimento):
    return [''.join(random.choice('01') for _ in range(comprimento)) for _ in range(tamanho)]

def selecionar(populacao, alvo):
    """Seleção por torneio (2 indivíduos)."""
    a, b = random.sample(populacao, 2)
    return a if fitness(a, alvo) < fitness(b, alvo) else b

def cruzamento(pai1, pai2):
    ponto = random.randint(1, len(pai1)-1)
    return pai1[:ponto] + pai2[ponto:]

def mutacao(individuo, taxa=0.05):
    return ''.join(
        str(1-int(g)) if random.random()<taxa else g
        for g in individuo
    )

# Evolução
ALVO     = "10110101001011"
POP_SIZE = 30
GERACOES = 100

populacao = criar_populacao(POP_SIZE, len(ALVO))
historico_fitness = []

for gen in range(GERACOES):
    populacao.sort(key=lambda x: fitness(x, ALVO))
    melhor_fit = fitness(populacao[0], ALVO)
    historico_fitness.append(melhor_fit)
    if melhor_fit == 0: break
    nova_pop = populacao[:2]  # elitismo
    while len(nova_pop) < POP_SIZE:
        p1, p2 = selecionar(populacao, ALVO), selecionar(populacao, ALVO)
        filho = cruzamento(p1, p2)
        filho = mutacao(filho)
        nova_pop.append(filho)
    populacao = nova_pop

plt.figure(figsize=(8,3))
plt.plot(historico_fitness, 'g-')
plt.xlabel('Geração'); plt.ylabel('Fitness (erros)')
plt.title(f'Algoritmo Genético — Evoluindo para "{ALVO}"'); plt.grid(True); plt.show()

print(f"Alvo:         {ALVO}")
print(f"Melhor indiv: {populacao[0]}")
print(f"Fitness:      {fitness(populacao[0], ALVO)} erros restantes")


### 📝 Exercício 3

Modifique o algoritmo genético para evoluir para uma **string de texto** (não só 0s e 1s). Use o alfabeto completo (a-z, espaço). Evolua para seu nome!

In [ ]:
import string

def fitness_texto(individuo, alvo):
    return sum(a!=b for a,b in zip(individuo,alvo))

def criar_individuo_texto(comprimento):
    alfabeto = string.ascii_lowercase + ' '
    return ''.join(random.choice(alfabeto) for _ in range(comprimento))

# ✏️ Mude o alvo para seu nome (minúsculas)
ALVO_TEXTO = "inteligencia artificial"
POP_SIZE2 = 100
GERACOES2 = 500

# TODO: adapte o loop evolutivo para strings de texto
# Dica: mutação agora deve escolher aleatoriamente entre os caracteres do alfabeto


## 4. Comparação dos Paradigmas

### 📝 Exercício Final — Complete a tabela

| Paradigma | Exemplo | Vantagem | Limitação |
|-----------|---------|----------|-----------|
| Simbólico | Regras IF-THEN | *(preencha)* | *(preencha)* |
| Conexionista | Redes Neurais | *(preencha)* | *(preencha)* |
| Evolucionário | Algoritmos Genéticos | *(preencha)* | *(preencha)* |

*(Edite esta célula Markdown para completar a tabela)*